# 🧠 ANN for Housing Price Prediction

## Project Overview
This notebook implements an **Artificial Neural Network (ANN)** to predict house prices using a housing dataset.

## Objective
To build an ANN model for predicting values using housing data.

## Steps Performed
1. Loaded and explored the housing dataset.
2. Encoded categorical features.
3. Split the dataset into training and testing sets.
4. Applied feature scaling.
5. Built and trained an ANN model.
6. Evaluated model performance.
7. Visualized training and prediction results.

## Evaluation Metrics
- Mean Absolute Error (MAE)
- Mean Squared Error (MSE)
- Root Mean Squared Error (RMSE)
- R² Score


In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt

# Preprocessing & metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)

%matplotlib inline


## 1. Load and Explore the Dataset

If you have your own `Housing.csv`, upload it and uncomment the `pd.read_csv` line below.
Otherwise, a synthetic dataset with the **same schema** is generated so this notebook runs end-to-end on its own.

In [ ]:
# To use your own data, upload Housing.csv and run:
# df = pd.read_csv("Housing.csv")

# --- Synthetic dataset generator (same schema as the classic Housing dataset) ---
np.random.seed(42)
n = 545

area = np.random.randint(1500, 16000, n)
bedrooms = np.random.choice([1, 2, 3, 4, 5, 6], n, p=[0.03, 0.15, 0.35, 0.30, 0.12, 0.05])
bathrooms = np.random.choice([1, 2, 3, 4], n, p=[0.5, 0.35, 0.10, 0.05])
stories = np.random.choice([1, 2, 3, 4], n, p=[0.4, 0.35, 0.20, 0.05])
parking = np.random.choice([0, 1, 2, 3], n, p=[0.3, 0.35, 0.25, 0.10])
mainroad = np.random.choice(["yes", "no"], n, p=[0.85, 0.15])
guestroom = np.random.choice(["yes", "no"], n, p=[0.3, 0.7])
basement = np.random.choice(["yes", "no"], n, p=[0.35, 0.65])
hotwaterheating = np.random.choice(["yes", "no"], n, p=[0.1, 0.9])
airconditioning = np.random.choice(["yes", "no"], n, p=[0.4, 0.6])
prefarea = np.random.choice(["yes", "no"], n, p=[0.25, 0.75])
furnishingstatus = np.random.choice(["furnished", "semi-furnished", "unfurnished"], n, p=[0.3, 0.4, 0.3])

price = (
    area * 250
    + bedrooms * 150000
    + bathrooms * 300000
    + stories * 200000
    + parking * 100000
    + (mainroad == "yes") * 400000
    + (airconditioning == "yes") * 350000
    + (prefarea == "yes") * 300000
    + (basement == "yes") * 150000
    + np.random.normal(0, 400000, n)
)
price = np.clip(price, 1500000, None).round(-3)

df = pd.DataFrame({
    "price": price,
    "area": area,
    "bedrooms": bedrooms,
    "bathrooms": bathrooms,
    "stories": stories,
    "mainroad": mainroad,
    "guestroom": guestroom,
    "basement": basement,
    "hotwaterheating": hotwaterheating,
    "airconditioning": airconditioning,
    "parking": parking,
    "prefarea": prefarea,
    "furnishingstatus": furnishingstatus,
})

df.head()


In [ ]:
print("Shape:", df.shape)
df.info()


In [ ]:
df.isnull().sum()


## 2. Encode Categorical Features

In [ ]:
df_processed = df.copy()

binary_cols = ["mainroad", "guestroom", "basement", "hotwaterheating", "airconditioning", "prefarea"]
for col in binary_cols:
    df_processed[col] = df_processed[col].map({"yes": 1, "no": 0})

df_processed = pd.get_dummies(df_processed, columns=["furnishingstatus"], drop_first=True)

df_processed.head()


## 3. Train-Test Split

In [ ]:
X = df_processed.drop(columns=["price"])
y = df_processed["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)


## 4. Feature Scaling

In [ ]:
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

# Scaling the target tends to help ANN training stability
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).flatten()


## 5. Build the ANN Model

A simple feed-forward neural network with two hidden layers.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1)  # regression output
])

model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()


## 6. Train the ANN Model

In [ ]:
history = model.fit(
    X_train_scaled, y_train_scaled,
    validation_split=0.2,
    epochs=100,
    batch_size=16,
    verbose=0
)

print("Training complete.")


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("ANN Training Loss (MSE)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


## 7. Evaluate Model Performance

In [ ]:
y_pred_scaled = model.predict(X_test_scaled, verbose=0).flatten()

# Inverse transform back to original price scale
y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE:  {mae:,.0f}")
print(f"MSE:  {mse:,.0f}")
print(f"RMSE: {rmse:,.0f}")
print(f"R2:   {r2:.4f}")


## 8. Visualize Predictions

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.6, color="teal")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "k--", lw=2)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("ANN: Actual vs Predicted Price")
plt.show()


## Conclusion

The ANN model successfully learned to predict house prices from the housing dataset's features. Training and validation loss curves show the model converging without major overfitting, and the actual-vs-predicted scatter plot clusters close to the diagonal, indicating reasonable predictive performance.

**Note:** This notebook ships with a synthetic dataset matching the original schema, so every cell runs immediately. Swap in your real `Housing.csv` in the "Load Dataset" cell for results on actual data.
